Este cuaderno, como su nombre indica, es para realizar el entrenamiento y evaluación de modelos de vídeo.

Los modelos que vamos a trabajar en este cuaderno son los siguientes:
- *MCG-NJU/videomae-base* --> inspirado en cómo aprender los modelos de lenguaje (escondiendo palabras para que el modelo las adivine). Este modelo, toma un video, oculta el 90% de los píxeles (en forma de cubosde espacio-tiempo) y se fuerza a sí mismo a reconstruir lo que falta viendo solo el 10% restante.
- *facebook/timesformer-base-finetuned-k400* --> este modelo soluciona un problema crítico en este tipo de tareas: aplicar la atención (heredado de los transformers) sin tener que hacerlo a cada píxel (lo cual consume mucha memoria). Este modelo primero mira la relación espacial (píxeles dentro de un mismo frame) y luego la relación temporal (el mismo píxel a lo largo de los diferentes frames).
- *google/vivit-b-16x2-kinetics400* --> es la evolución directa de un ViT clásico, los cuales los hemos trabajado en el apartado anterior. Este modelo divide el vídeo en "Tubelets" o tubos 3D. Extrae un bloque de píxeles que atraviesa varios frames de golpe desde la primera capa.  

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
import decord
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import evaluate
from transformers import (
    AutoImageProcessor,
    AutoModelForVideoClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# 1. Selección del modelo y parámetros

In [ ]:
# Descomentar el modelo que se quiera entrenar:
MODELO_ELEGIDO = "videomae"
#MODELO_ELEGIDO = "timesformer"
#MODELO_ELEGIDO = "vivit"

In [ ]:
# Diccionario de Checkpoints (Hugging Face)
MODEL_ZOO = {
    "videomae": "MCG-NJU/videomae-base",
    "timesformer": "facebook/timesformer-base-finetuned-k400",
    "vivit": "google/vivit-b-16x2-kinetics400"
}